In [1]:
import os
import torch
os.environ['TORCH'] = torch.__version__
print(torch.__version__)

!pip install pyg-lib -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install git+https://github.com/pyg-team/pytorch_geometric.git

2.10.0+cpu
Looking in links: https://data.pyg.org/whl/torch-2.10.0+cpu.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 24.3 MB/s eta 0:00:00
  Cloning https://github.com/pyg-team/pytorch_geometric.git to /tmp/pip-req-build-z9yap3t8
  Running command git clone --filter=blob:none --quiet https://github.com/pyg-team/pytorch_geometric.git /tmp/pip-req-build-z9yap3t8
  Resolved https://github.com/pyg-team/pytorch_geometric.git to commit fae65964d712b1c93d620e23237a18b2f3ffaa9e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for torch-geometric: filename=torch_geometric-2.8.0-py3-none-any.whl size=1323437 sha256=4a4a8cd34562ec2c77e0b62ce992330e5fe2ddfef21075ca4b13a2aea9bb89d2
  Stored in directory: /tmp/pip-ephem-wheel-cache-h3t2hek1/wheels/96/ab/80/5e43250505a6e639df59a3d89c6b45ed5511f70db8d0ac39c7
Successfully built torch-geometric


In [2]:
import torch
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import NormalizeFeatures
dataset = Planetoid(root='data/Planetoid', name='PubMed', transform=NormalizeFeatures())
print()
print(f'Dataset: {dataset}:')
print('==================')
print(f'Number of graphs: {len(dataset)}')
print(f'Number of features: {dataset.num_features}')
print(f'Number of classes: {dataset.num_classes}')
data = dataset[0]  # Get the first graph object.
print()
print(data)
print('===============================================================================================================')
# Gather some statistics about the graph.
print(f'Number of nodes: {data.num_nodes}')
print(f'Number of edges: {data.num_edges}')
print(f'Average node degree: {data.num_edges / data.num_nodes:.2f}')
print(f'Number of training nodes: {data.train_mask.sum()}')
print(f'Training node label rate: {int(data.train_mask.sum()) / data.num_nodes:.3f}')
print(f'Has isolated nodes: {data.has_isolated_nodes()}')
print(f'Has self-loops: {data.has_self_loops()}')
print(f'Is undirected: {data.is_undirected()}')

Processing...



Dataset: PubMed():
Number of graphs: 1
Number of features: 500
Number of classes: 3

Data(x=[19717, 500], edge_index=[2, 88648], y=[19717], train_mask=[19717], val_mask=[19717], test_mask=[19717])
Number of nodes: 19717
Number of edges: 88648
Average node degree: 4.50
Number of training nodes: 60
Training node label rate: 0.003
Has isolated nodes: False
Has self-loops: False
Is undirected: True


Done!


In [5]:
from torch_geometric.loader import ClusterData, ClusterLoader

torch.manual_seed(12345)
cluster_data = ClusterData(data, num_parts=128)  # 1. Create subgraphs.
train_loader = ClusterLoader(cluster_data, batch_size=32, shuffle=True)  # 2. Stochastic partioning scheme.

print()
total_num_nodes = 0
for step, sub_data in enumerate(train_loader):
    print(f'Step {step + 1}:')
    print('=======')
    print(f'Number of nodes in the current batch: {sub_data.num_nodes}')
    print(sub_data)
    print()
    total_num_nodes += sub_data.num_nodes

print(f'Iterated over {total_num_nodes} of {data.num_nodes} nodes!')


Step 1:
Number of nodes in the current batch: 4930
Data(x=[4930, 500], y=[4930], train_mask=[4930], val_mask=[4930], test_mask=[4930], edge_index=[2, 16814])

Step 2:
Number of nodes in the current batch: 4930
Data(x=[4930, 500], y=[4930], train_mask=[4930], val_mask=[4930], test_mask=[4930], edge_index=[2, 16810])

Step 3:
Number of nodes in the current batch: 4937
Data(x=[4937, 500], y=[4937], train_mask=[4937], val_mask=[4937], test_mask=[4937], edge_index=[2, 16996])

Step 4:
Number of nodes in the current batch: 4920
Data(x=[4920, 500], y=[4920], train_mask=[4920], val_mask=[4920], test_mask=[4920], edge_index=[2, 16086])

Iterated over 19717 of 19717 nodes!


Computing METIS partitioning...
Done!


In [6]:
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class GCN(torch.nn.Module):
    def __init__(self, hidden_channels):
        super(GCN, self).__init__()
        torch.manual_seed(12345)
        self.conv1 = GCNConv(dataset.num_node_features, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, dataset.num_classes)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = x.relu()
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x

model = GCN(hidden_channels=16)
print(model)

GCN(
  (conv1): GCNConv(500, 16)
  (conv2): GCNConv(16, 3)
)


In [7]:
from IPython.display import Javascript
display(Javascript('''google.colab.output.setIframeHeight(0, true, {maxHeight: 300})'''))

model = GCN(hidden_channels=16)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
criterion = torch.nn.CrossEntropyLoss()

def train():
      model.train()

      for sub_data in train_loader:  # Iterate over each mini-batch.
          out = model(sub_data.x, sub_data.edge_index)  # Perform a single forward pass.
          loss = criterion(out[sub_data.train_mask], sub_data.y[sub_data.train_mask])  # Compute the loss solely based on the training nodes.
          loss.backward()  # Derive gradients.
          optimizer.step()  # Update parameters based on gradients.
          optimizer.zero_grad()  # Clear gradients.

def test():
      model.eval()
      out = model(data.x, data.edge_index)
      pred = out.argmax(dim=1)  # Use the class with highest probability.

      accs = []
      for mask in [data.train_mask, data.val_mask, data.test_mask]:
          correct = pred[mask] == data.y[mask]  # Check against ground-truth labels.
          accs.append(int(correct.sum()) / int(mask.sum()))  # Derive ratio of correct predictions.
      return accs

for epoch in range(1, 51):
    loss = train()
    train_acc, val_acc, test_acc = test()
    print(f'Epoch: {epoch:03d}, Train: {train_acc:.4f}, Val Acc: {val_acc:.4f}, Test Acc: {test_acc:.4f}')

<IPython.core.display.Javascript object>

Epoch: 001, Train: 0.5833, Val Acc: 0.4960, Test Acc: 0.4890
Epoch: 002, Train: 0.7167, Val Acc: 0.5820, Test Acc: 0.5850
Epoch: 003, Train: 0.7000, Val Acc: 0.5940, Test Acc: 0.6170
Epoch: 004, Train: 0.8833, Val Acc: 0.7120, Test Acc: 0.6990
Epoch: 005, Train: 0.8833, Val Acc: 0.6980, Test Acc: 0.6870
Epoch: 006, Train: 0.9333, Val Acc: 0.7320, Test Acc: 0.7260
Epoch: 007, Train: 0.9500, Val Acc: 0.7660, Test Acc: 0.7410
Epoch: 008, Train: 0.9500, Val Acc: 0.7620, Test Acc: 0.7560
Epoch: 009, Train: 0.9667, Val Acc: 0.7640, Test Acc: 0.7540
Epoch: 010, Train: 0.9500, Val Acc: 0.7580, Test Acc: 0.7540
Epoch: 011, Train: 0.9333, Val Acc: 0.7440, Test Acc: 0.7400
Epoch: 012, Train: 0.9333, Val Acc: 0.7460, Test Acc: 0.7400
Epoch: 013, Train: 0.9500, Val Acc: 0.7660, Test Acc: 0.7490
Epoch: 014, Train: 0.9667, Val Acc: 0.7480, Test Acc: 0.7320
Epoch: 015, Train: 0.9667, Val Acc: 0.7500, Test Acc: 0.7240
Epoch: 016, Train: 0.9833, Val Acc: 0.7740, Test Acc: 0.7470
Epoch: 017, Train: 0.966